# Diff-MoE on BabyLM -- Kaggle T4 x2 training notebook

Runs one config from `docs/plan.md`'s 2x2 ablation (standard/differential attention x dense/MoE FFN) on Kaggle's dual-T4 notebook, using both GPUs via DDP. Training data is the [BabyLM Challenge](https://babylm.github.io/) corpus: six domains (child-directed speech, adult conversation, literary prose, subtitles, Wikipedia, telephone dialogue), released as one text file per domain rather than a single blob -- which is what makes it a meaningful testbed for whether MoE experts specialize.

**Before running**: set Notebook Settings -> Accelerator = **GPU T4 x2**.

**Session limits**: 12h max, ~30 GPU-h/week (same quota whether you pick T4 x1 or x2). Checkpoints save every `ckpt_freq` steps to `/kaggle/working/checkpoints` -- copy that folder to a Kaggle Dataset before your session ends so the next session can resume.

**Workflow per run**: (1) clone repo, (2) prepare tokenized data once and re-use across sessions via a Kaggle Dataset, (3) throughput probe on both GPUs, (4) train with DDP, (5) inspect metrics + report.

## 1. Setup

In [ ]:
!git clone -b rebuild https://github.com/ramprasathk07/Differential-MOE.git /kaggle/working/repo
%cd /kaggle/working/repo
!pip install -q -r requirements.txt

In [ ]:
# Staleness guard: Kaggle keeps its OWN copy of this notebook, independent of the
# repo. If either side is behind, cells pass flags or name configs the other side
# doesn't have -- which surfaces later as a confusing FileNotFoundError on
# train.bin rather than as "your notebook is out of date". Check both directions
# up front, while it is still cheap to fix.
import glob
import subprocess

_help = subprocess.run(['python', '-m', 'src.data.train_tokenizer', '--help'],
                       capture_output=True, text=True).stdout
_missing = []
if '--track' not in _help:
    _missing.append('src.data.train_tokenizer has no --track (pre-BabyLM code)')
if '--pretrained' not in _help:
    _missing.append('src.data.train_tokenizer has no --pretrained (pre-tier-S code)')
if not glob.glob('configs/s_*.yaml'):
    _missing.append('configs/s_*.yaml absent (tier S configs missing)')

if _missing:
    raise SystemExit(
        'STALE CODE/NOTEBOOK MISMATCH:\n  - ' + '\n  - '.join(_missing) + '\n\n'
        'The clone is behind this notebook. Fix whichever applies:\n'
        '  * the clone cell points at a branch without these changes -- check -b <branch>\n'
        '  * /kaggle/working/repo is a stale checkout from an earlier session --\n'
        '    `rm -rf /kaggle/working/repo` and re-run the clone cell\n'
        '  * or this notebook is newer than the pushed code -- push first'
    )
print('OK: cloned code matches this notebook (--track, --pretrained, tier-S configs all present).')
print('tier S configs found:', sorted(glob.glob('configs/s_*.yaml')))

In [ ]:
import torch
n_gpu = torch.cuda.device_count()
print('CUDA available:', torch.cuda.is_available())
print('GPU count:', n_gpu)
for i in range(n_gpu):
    print(f'  cuda:{i}', torch.cuda.get_device_name(i))
if n_gpu < 2:
    print('\nWARNING: fewer than 2 GPUs visible -- set Notebook Settings > Accelerator = GPU T4 x2, '
          'then Session > Restart & Run All. The --ddp cells below need nproc_per_node=2 to match n_gpu.')

## 2. Run settings

Change these per run -- no code edits needed elsewhere in the notebook.

In [ ]:
import os

# --- pick ONE config -------------------------------------------------------
# tier A  (~16M, custom 4k BPE):  a_dense / a_diff / a_moe / a_diffmoe
# tier B  (~55M, custom 8k BPE):  b_final
# tier S (~295M, cl100k frontier): s_dense / s_diff / s_moe / s_diffmoe
CONFIG = 'configs/s_dense.yaml'

WANDB_PROJECT = 'diff-moe-kaggle'  # change freely per experiment batch
USE_WANDB = True
N_GPU = 2  # matches Accelerator = GPU T4 x2; set to 1 for the single-T4 option

_name = CONFIG.split('/')[-1].replace('.yaml', '')
TIER = _name[0]  # 'a', 'b' or 's'

# Tier S uses a pretrained frontier tokenizer (no BPE training step) and the
# 100M-word 'strict' track; tiers A/B train a custom BPE on 'strict-small'.
if TIER == 's':
    TRACK, DATA_DIR = 'strict', '/kaggle/working/data_s'
    TOKENIZER = 'hf:Xenova/gpt-4'   # cl100k. For Qwen: 'hf:Qwen/Qwen2.5-0.5B'
                                     #   -> then set vocab_size: 151680 in the s_*.yaml configs
else:
    TRACK = 'strict' if TIER == 'b' else 'strict-small'
    DATA_DIR = '/kaggle/working/data_b' if TIER == 'b' else '/kaggle/working/data'
    TOKENIZER = None  # trained below by the sweep + train step

# wandb auth: without this, rank0's wandb.init() blocks at a login prompt inside
# the non-interactive training cell. Add-ons > Secrets > WANDB_API_KEY.
if USE_WANDB:
    try:
        from kaggle_secrets import UserSecretsClient
        os.environ['WANDB_API_KEY'] = UserSecretsClient().get_secret('WANDB_API_KEY')
        print('wandb key loaded from Kaggle secret WANDB_API_KEY')
    except Exception as e:
        print('WARNING: no WANDB_API_KEY Kaggle secret -- the training cell will hang at wandb login.')
        print('Fix: Add-ons > Secrets > add WANDB_API_KEY, or set USE_WANDB = False. Details:', e)

run_name = _name
wandb_flag = f'--wandb --wandb_project {WANDB_PROJECT}' if USE_WANDB else ''
ddp_prefix = f'torchrun --standalone --nproc_per_node={N_GPU}' if N_GPU > 1 else 'python'
ddp_flag = '--ddp' if N_GPU > 1 else ''
print(f'run_name: {run_name} | tier: {TIER} | track: {TRACK}')
print(f'data_dir: {DATA_DIR} | tokenizer: {TOKENIZER or "custom BPE (trained below)"}')
print('launch prefix:', ddp_prefix)

## 3. Tokenizer + data

Run once, then attach the output as a Kaggle Dataset ("New Dataset" from notebook output) so later sessions can skip straight to training via `DATA_DIR` pointing at `/kaggle/input/<dataset-name>`. Tokenizing is CPU-only -- no benefit from N_GPU here. Downloads the six BabyLM domain files (bnc_spoken, childes, gutenberg, open_subtitles, simple_wiki, switchboard) for whichever `TRACK` the settings cell selected.

In [ ]:
import os

# Tier A/B: adaptive vocab sweep (docs/plan.md SS2) -- read the fertility table,
# pick a vocab_size, set it below. BabyLM's six domains are heterogeneous, so
# don't reuse a vocab size chosen for a different corpus.
# Tier S: skipped -- it uses a pretrained frontier tokenizer, nothing to train.
# The --pretrained flag scores them side by side, which is what justifies the
# choice (lower fertility, but far more embedding params -- see docs/plan.md).
if TOKENIZER is not None:
    print(f'tier S: using pretrained {TOKENIZER}, no BPE training needed.')
    print('(optional) compare it against custom vocabs on this corpus:')
    print(f'  !python -m src.data.train_tokenizer --sweep --candidates 4096 16384 '
          f'--pretrained {TOKENIZER} --track {TRACK} --max_lines_per_domain 20000')
elif not os.path.exists(f'{DATA_DIR}/train.bin'):
    !python -m src.data.train_tokenizer --sweep --candidates 2048 4096 8192 16384 --track {TRACK} --max_lines_per_domain 20000
else:
    print('data already prepared at', DATA_DIR)

In [ ]:
VOCAB_SIZE = 4096  # tier A/B only: set from the sweep table above; must match CONFIG's vocab_size

# Tokenizing the full 'strict' track (100M words) with a frontier tokenizer takes
# a while and writes uint32 .bin files (2x the size of uint16). Do it once, then
# save /kaggle/working as a Dataset and point DATA_DIR at /kaggle/input/<name>.
if not os.path.exists(f'{DATA_DIR}/train.bin'):
    if TOKENIZER is not None:                      # tier S: pretrained, no training step
        !python -m src.data.prepare --tokenizer {TOKENIZER} --out_dir {DATA_DIR} --track {TRACK}
    else:                                          # tier A/B: train the custom BPE first
        !python -m src.data.train_tokenizer --vocab_size {VOCAB_SIZE} --out {DATA_DIR}/tokenizer.json --track {TRACK}
        !python -m src.data.prepare --tokenizer {DATA_DIR}/tokenizer.json --out_dir {DATA_DIR} --track {TRACK}
else:
    print('data already prepared at', DATA_DIR)

# prepare.py prints the vocab it saw and records it in meta.json; train.py hard-fails
# if CONFIG's vocab_size can't cover it, so a mismatch can't waste a paid run.
import json
if os.path.exists(f'{DATA_DIR}/meta.json'):
    print(json.dumps(json.load(open(f'{DATA_DIR}/meta.json')), indent=2))

## 4. Throughput probe (docs/plan.md Phase 3)

Runs ~100 steps across both GPUs and reports tok/s, so the token budget and wall-clock are **measured, not guessed**. `batch_size` in the config is **per-GPU**, so with `N_GPU=2` the effective global batch doubles automatically (see `effective_global_batch_tokens` in `report.json`).

For tier S this probe is doing real work beyond timing — it is the first thing that would OOM. At vocab 100k the fp32 logits tensor is `batch x seq x vocab x 4B`, which is why `s_*.yaml` ships a small `batch_size: 4`. If the probe survives with memory to spare, you can raise `batch_size` and halve `accum_steps` to cut step overhead; if it OOMs, halve `batch_size` and double `accum_steps` (global batch stays constant either way).

The cell after the probe converts measured throughput into a concrete `max_steps` for your quota — **use that number**, not the 2900 placeholder in the config, which came from an unvalidated 25 TFLOP/s estimate.

In [ ]:
# NOTE: probe writes to its own out_dir so its checkpoints / LR history / 'best'
# selections don't pollute the real run (which would otherwise auto-resume from
# the probe's step-100 checkpoint with a mismatched cosine schedule).
!{ddp_prefix} -m src.train --config {CONFIG} --data_dir {DATA_DIR} --out_dir /kaggle/working/probe {ddp_flag} --max_steps 100

In [ ]:
# Turn the probe's measured tok/s into a max_steps you can actually afford.
# Reads the probe's own metrics.csv rather than asking you to eyeball the log.
import pandas as pd, yaml, json, os

HOURS_PER_RUN = 7.5   # 30h weekly quota / 4 ablation runs
N_RUNS = 4

probe_csv = f'/kaggle/working/probe/{run_name}/metrics.csv'
cfg = yaml.safe_load(open(CONFIG))
tok_per_step = (cfg['train']['batch_size'] * cfg['model']['seq_len']
                * cfg['train']['accum_steps'] * N_GPU)

if os.path.exists(probe_csv):
    df = pd.read_csv(probe_csv)
    rate = df['tok_per_sec'].dropna()
    # drop the first logged point: it carries warmup/compile cost
    measured = rate.iloc[1:].median() if len(rate) > 1 else rate.iloc[0]
    affordable = int(measured * HOURS_PER_RUN * 3600 / tok_per_step)

    meta = json.load(open(f'{DATA_DIR}/meta.json')) if os.path.exists(f'{DATA_DIR}/meta.json') else {}
    train_tokens = meta.get('tokens', {}).get('train')

    print(f'measured        : {measured:,.0f} tok/s  ({tok_per_step:,} tok/step)')
    print(f'config max_steps: {cfg["train"]["max_steps"]}  '
          f'-> {cfg["train"]["max_steps"]*tok_per_step/3600/measured:.1f} h')
    print(f'affordable      : {affordable} steps in {HOURS_PER_RUN}h '
          f'({affordable*tok_per_step/1e6:,.0f}M tokens)')
    if train_tokens:
        print(f'                  = {affordable*tok_per_step/train_tokens:.1f} epochs of this corpus '
              f'({train_tokens/1e6:.0f}M tokens)')
        print('  >4 epochs starts to repeat data heavily; consider fewer steps or more data.')
    print(f'\nquota check     : {N_RUNS} runs x {HOURS_PER_RUN}h = {N_RUNS*HOURS_PER_RUN}h')
    print(f'\n-> set max_steps: {affordable} in {CONFIG} before the full run')
else:
    print('run the probe cell first')

## 5. Full training run

Re-running this cell auto-resumes from `checkpoints/<run_name>/last.pt` if it exists -- safe to re-run after a Kaggle session restart. Rank-0-only logging/checkpointing/wandb is handled inside `src/train.py`; nothing extra needed here.

In [ ]:
!{ddp_prefix} -m src.train --config {CONFIG} --data_dir {DATA_DIR} --out_dir /kaggle/working/checkpoints {ddp_flag} {wandb_flag}

## 6. Inspect metrics + report

In [ ]:
import json
with open(f'/kaggle/working/checkpoints/{run_name}/report.json') as f:
    print(json.dumps(json.load(f), indent=2))

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv(f'/kaggle/working/checkpoints/{run_name}/metrics.csv')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df.dropna(subset=['train_loss']).plot(x='step', y='train_loss', ax=axes[0], title='train loss')
df.dropna(subset=['val_nll']).plot(x='step', y='val_nll', ax=axes[1], title='val NLL', marker='o')
plt.tight_layout()
plt.show()
df.tail(10)

## 7. Params table (for README / blog tables -- never hand-compute)

In [ ]:
# Never hand-compute these for a README/blog table -- regenerate them.
# Confirms the two parity claims the ablation rests on: standard vs differential
# attention cost the same, and dense vs MoE cost the same *active* params.
print('--- tier A (~16M, custom 4k BPE) ---')
!python -m src.params --config configs/a_dense.yaml configs/a_diff.yaml configs/a_moe.yaml configs/a_diffmoe.yaml
print('\n--- tier S (~295M active, cl100k) ---')
!python -m src.params --config configs/s_dense.yaml configs/s_diff.yaml configs/s_moe.yaml configs/s_diffmoe.yaml

## 8. Persist checkpoints across sessions

Kaggle wipes `/kaggle/working` between sessions. 'Save Version' preserves everything under `/kaggle/working` as this notebook's output. To resume in a NEXT session: attach that output (or a Dataset made from it) as an input, then **copy it back into `/kaggle/working`** before training -- `/kaggle/input` is read-only, so pointing `--out_dir` at it directly would crash on the first checkpoint save.

In [ ]:
# This session's checkpoints (saved automatically when you 'Save Version'):
!ls -la /kaggle/working/checkpoints/{run_name}/ 2>/dev/null || echo 'no checkpoints yet'
print("best/ holds only the top-2 checkpoints (max_best_checkpoints in config); last.pt is always kept for resume.")

# NEXT session, to resume: attach the previous version's output as an input dataset,
# then copy it into the writable working dir BEFORE running the training cell
# (/kaggle/input is READ-ONLY -- training must never write there). Uncomment + edit:
# !mkdir -p /kaggle/working/checkpoints
# !cp -r /kaggle/input/<your-ckpt-dataset>/checkpoints/* /kaggle/working/checkpoints/